# **Updating active listings**

---

In [ ]:
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os
import pandas as pd
from sqlalchemy import text

load_dotenv("../shhh.env")
engine_connection = os.getenv("SQL_ENGINE")
engine = create_engine(engine_connection)

## *Listings on the Given Date*

Checking how many scrapings happened on the given date, meaning these listings were on the site for sale 

Since the scraper checks every listings that means the other listings in the database were not seen

In [6]:
DATE = "2025-09-06"

query = text("""
    SELECT COUNT(*) AS car_count
    FROM car_data
    WHERE DATE(last_seen) = :date_value
""")

car_count = pd.read_sql_query(query, engine, params={"date_value": DATE})
display(car_count.style.hide(axis=0))

car_count
92509


## *Deactivating*

Deactivating cars which weren't scraped (= seen) on the given date, meaning they were either sold or removed from the site

Unfortunetly that is not know if they were actually bought or just removed

In [ ]:
query = text(f"""
    UPDATE car_data
    SET active = false
    WHERE DATE(last_seen) != '{DATE}'
""")

with engine.begin() as conn:
    conn.execute(query, {"date_value": DATE})


## *Verifying The Database Change*

In [7]:
check_query = """
    SELECT last_seen, active, COUNT(*) as count
    FROM car_data
    GROUP BY last_seen, active
    ORDER BY last_seen DESC
"""
result = pd.read_sql_query(check_query, engine)
display(result)

,last_seen,active,count
0,2025-09-06,True,92509
1,2025-09-05,False,1425
2,2025-08-23,False,25926
3,2025-08-22,False,319
4,2025-08-21,False,973
5,2025-08-12,False,873
6,2025-08-11,False,19597
7,2025-08-06,False,1135
8,2025-08-05,False,1951
9,2025-08-04,False,3825
